In [4]:
import re
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
# Step 1: Define folder paths
from pathlib import Path

# Move one level up from 'scripts/' to reach project root
BASE_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
TRAIN_PATH = BASE_DIR / "data" / "raw" / "train.csv"

print(f"Loading data from: {TRAIN_PATH}")
print("--- Step 1: Loading Raw Data ---")
df = pd.read_csv(TRAIN_PATH)

# Let's inspect the data shape and first few rows
print(f"Total rows and columns: {df.shape}")
print(df.head(3))

Loading data from: /home/dell/Documents/Final_project_Car_Insurance_Claim_Prediction/data/raw/train.csv
--- Step 1: Loading Raw Data ---
Total rows and columns: (58592, 44)
  policy_id  policy_tenure  age_of_car  age_of_policyholder area_cluster  \
0   ID00001       0.515874        0.05             0.644231           C1   
1   ID00002       0.672619        0.02             0.375000           C2   
2   ID00003       0.841110        0.02             0.384615           C3   

   population_density  make segment model fuel_type  ... is_brake_assist  \
0                4990     1       A    M1       CNG  ...              No   
1               27003     1       A    M1       CNG  ...              No   
2                4076     1       A    M1       CNG  ...              No   

  is_power_door_locks is_central_locking  is_power_steering  \
0                  No                 No                Yes   
1                  No                 No                Yes   
2                  No       

In [6]:
# Step 2: Drop unused ID columns if present
if "policy_id" in df.columns:
    df = df.drop(columns=["policy_id"])

# Step 3: Extract numerical numbers from string columns (max_torque & max_power)
print("\n--- Step 2: Cleaning Torque and Power Features ---")


--- Step 2: Cleaning Torque and Power Features ---


In [7]:
def extract_torque_value(text):
    """Helper function to extract torque (Nm) from strings like '113Nm@4400rpm'."""
    match = re.search(r"([\d\.]+)Nm", str(text))
    return float(match.group(1)) if match else np.nan


def extract_power_value(text):
    """Helper function to extract power (bhp) from strings like '88.50bhp@6000rpm'."""
    match = re.search(r"([\d\.]+)bhp", str(text))
    return float(match.group(1)) if match else np.nan


if "max_torque" in df.columns:
    df["torque_nm"] = df["max_torque"].apply(extract_torque_value)
    df = df.drop(columns=["max_torque"])

if "max_power" in df.columns:
    df["power_bhp"] = df["max_power"].apply(extract_power_value)
    df = df.drop(columns=["max_power"])

# Step 4: Convert binary 'Yes'/'No' columns to 1 and 0
print("\n--- Step 3: Encoding Binary Features ---")
binary_columns = [
    col for col in df.columns if col.startswith("is_") and col != "is_claim"
]

for col in binary_columns:
    if df[col].dtype == "object":
        df[col] = df[col].map({"Yes": 1, "No": 0})

# Log-transform right-skewed feature
if "population_density" in df.columns:
    df["population_density"] = np.log1p(df["population_density"])

# Step 5: Handle categorical features using One-Hot Encoding (pd.get_dummies)
print("\n--- Step 4: One-Hot Encoding Categorical Columns ---")
X = df.drop(columns=["is_claim"])
y = df["is_claim"]


--- Step 3: Encoding Binary Features ---

--- Step 4: One-Hot Encoding Categorical Columns ---


In [8]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(f"Categorical columns found: {categorical_cols}")

# Apply get_dummies
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Fill any missing values with median for numeric columns
X_encoded = X_encoded.fillna(X_encoded.median())

# Step 6: Split Data into Train and Validation sets (80% Train, 20% Val)
print("\n--- Step 5: Splitting Data into Train & Validation ---")
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

Categorical columns found: ['area_cluster', 'segment', 'model', 'fuel_type', 'engine_type', 'rear_brakes_type', 'transmission_type', 'steering_type']

--- Step 5: Splitting Data into Train & Validation ---


In [9]:
# Scale numeric features using StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns
)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)

# Step 7: Save preprocessed data and scaler for future use
print("\n--- Step 6: Saving Processed Artifacts ---")
saved_artifacts = {
    "X_train": X_train_scaled,
    "X_val": X_val_scaled,
    "y_train": y_train,
    "y_val": y_val,
    "scaler": scaler,
    "feature_names": X_encoded.columns.tolist(),
}

joblib.dump(saved_artifacts, PROCESSED_DIR / "split_data.pkl")
print(
    f"SUCCESS! Cleaned data saved to: {PROCESSED_DIR / 'split_data.pkl'}"
)


--- Step 6: Saving Processed Artifacts ---
SUCCESS! Cleaned data saved to: /home/dell/Documents/Final_project_Car_Insurance_Claim_Prediction/scripts/data/processed/split_data.pkl
